In [1]:
from pathlib import Path
import sys

root = Path.cwd()
while not (root / "src").is_dir() and root != root.parent:
    root = root.parent

if not (root / "src").is_dir():
    raise FileNotFoundError("Could not find the repository folder containing src/")

sys.path.insert(0, str(root))

In [2]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report
from src.data_load import load_data
from src.preprocess import preprocess

In [3]:
seed = 42
qubits = 4

X_train, X_test, y_train, y_test = load_data()
_, _, X_train_pca, X_test_pca, pca = preprocess(X_train, X_test, qubits)

In [ ]:
mm = MinMaxScaler(feature_range=(0, np.pi))
X_train_pca_scaled = mm.fit_transform(X_train_pca)
X_test_pca_scaled = np.clip(mm.transform(X_test_pca), 0, np.pi)


In [ ]:
param_grid = {
    "C": [0.1, 1, 10, 100, 1000],
    "gamma": [0.001, 0.01, 0.1, 1, 10, "scale"],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
search = GridSearchCV(SVC(kernel="rbf"), param_grid, cv=cv, scoring="f1_macro")
search.fit(X_train_pca_scaled, y_train)      # tuning sees ONLY the training data

print("Best settings:", search.best_params_)
print("Best CV score:", search.best_score_)

Best settings: {'C': 1000, 'gamma': 0.01}
Best CV score: 0.9695249687871226


In [ ]:
best = search.best_estimator_
pred = best.predict(X_test_pca_scaled)       # test set used once

print("Tuned RBF accuracy:", accuracy_score(y_test, pred))
print("Malignant recall:", recall_score(y_test, pred, pos_label=0))
print("F1 (macro):", f1_score(y_test, pred, average="macro"))
print(classification_report(y_test, pred, target_names=["malignant", "benign"]))

Tuned RBF accuracy: 0.9473684210526315
Malignant recall: 0.9285714285714286
F1 (macro): 0.9434523809523809
              precision    recall  f1-score   support

   malignant       0.93      0.93      0.93        42
      benign       0.96      0.96      0.96        72

    accuracy                           0.95       114
   macro avg       0.94      0.94      0.94       114
weighted avg       0.95      0.95      0.95       114

